[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_01_clustering_en_clase.ipynb)

# S09 · 01 — Segmentar huéspedes con clustering · versión en clase

⏱ 45 min en clase · Sprint 9 · M3 (UD 3.9, clustering)

**Contexto TurisData.** TurisData Canarias tiene 1 200 huéspedes y hasta ahora les manda a todos la misma newsletter. Dirección pregunta: *¿hay grupos de clientes parecidos entre sí, y merece la pena tratarlos de forma distinta?* Aquí **no hay etiquetas**: nadie nos ha dicho a qué grupo pertenece cada huésped. Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego **nosotros** interpretamos qué significa cada grupo.

**Este cuaderno tiene 2 ejercicios** (1 a 2), los que se hacen en el aula. La interpretación de los grupos, el clustering jerárquico, DBSCAN, el mini-reto y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1 y 2 son los 2 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
print(huespedes.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Las primeras filas de huespedes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas columnas tiene la tabla y cuáles son numéricas?
2. ¿Qué variable solo puede valer 0 ó 1?
3. ¿Hay alguna columna que diga a qué grupo pertenece el huésped?

In [ ]:
huespedes.head()

### 👀 A3 · EJEMPLO — Las escalas de las variables

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Entre qué valores va `gasto_medio_dia`?
2. ¿Qué variable varía más en unidades absolutas y cuál menos?
3. Si agrupamos por distancia sin tocar estas escalas, ¿qué variables mandarían?

In [ ]:
huespedes.describe().round(1).T

### 👀 A4 · EJEMPLO — Estandarizar una sola columna

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué media y qué desviación tiene la edad estandarizada?
2. Un huésped de 53 años (la media es 41,2 y la desviación 12,1), ¿cuánto vale su edad escalada?
3. ¿Qué consigue esto para el clustering?

In [ ]:
edad = huespedes["edad"]
edad_escalada = (edad - edad.mean()) / edad.std()
print("media:", round(edad_escalada.mean(), 3), "· desviación:", round(edad_escalada.std(), 3))

### ✏️ A5 · EJERCICIO 1 de 2 — La matriz X estandarizada

**Tu turno:** Crea la matriz `X` con **todas las variables menos `id_huesped`**, estandarizadas con `StandardScaler` (usa `fit_transform`). `X` debe ser una matriz de 1200 filas y 7 columnas. Guarda también los nombres de las variables en la lista `variables`. Ojo: más abajo (apartado 3) se añade a la tabla la columna `grupo`; si repites esta celda después, **no la incluyas** en `variables` (o reinicia el cuaderno).

**Debe salir:** ✔ Ejercicio 1 correcto (`X` tiene 1200 filas y 7 columnas, todas con media 0 y desviación 1)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Por qué se quita `id_huesped`?
2. ¿Qué tipo de objeto es `X` y qué ha cambiado respecto a `huespedes`?
3. ¿Qué media y desviación tiene cada columna de `X`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: quita la columna id_huesped con drop(columns=...) y pásale el resultado a StandardScaler().fit_transform
variables = ...
X = ...

assert X.shape == (1200, 7), "X debe tener 1200 filas y 7 columnas (sin id_huesped ni grupo)"
assert abs(X.mean()) < 1e-6 and abs(X[:, 0].std() - 1) < 1e-6, "Revisa que estés escalando con StandardScaler"
print("✔ Ejercicio 1 correcto")

### 👀 A6 · EJEMPLO — k-means con k = 3

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos huéspedes hay en cada uno de los tres grupos?
2. ¿Qué decide el usuario en k-means y qué decide el algoritmo?
3. ¿Por qué se pone `random_state=SEMILLA`?

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)
print("Tamaño de cada grupo:", np.bincount(km3.labels_))
print("Inercia (suma de distancias al centro):", round(km3.inertia_))

### ✏️ A7 · EJERCICIO 2 de 2 — ¿Cuántos grupos? La silueta de cada k

**Tu turno:** Calcula en el diccionario `siluetas` la silueta media (`silhouette_score(X, etiquetas)`) de k-means para cada `k` de 2 a 8 (clave = `k`, valor = silueta). Después guarda en `k_mejor` el `k` con silueta más alta. Usa `n_init=10` y `random_state=SEMILLA`.

**Debe salir:** `k con mejor silueta: 4` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué k tiene la mejor silueta y cuánto vale?
2. ¿Cómo de buena es esa estructura según la escala del texto?
3. ¿Por qué `max(siluetas)` no daría el mejor k?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: bucle for k in range(2, 9): entrena KMeans, saca .labels_ y calcula silhouette_score; luego max(siluetas, key=siluetas.get)
siluetas = {}
k_mejor = ...

assert set(siluetas) == set(range(2, 9)), "siluetas debe tener las claves 2, 3, ..., 8"
assert 3 <= k_mejor <= 5, "Revisa: k_mejor debería ser el k con la silueta más alta (está entre 3 y 5)"
print("k con mejor silueta:", k_mejor)
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Dibujar la silueta de cada k

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿En qué k está el máximo de la curva?
2. ¿Con 3, 4 o 5 grupos las siluetas son muy distintas?
3. ¿Qué pasa a partir de k = 6?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(siluetas), list(siluetas.values()), marker="o")
ax.axvline(k_mejor, color="tab:red", linestyle="--", label=f"mejor k = {k_mejor}")
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta media")
ax.set_title("¿Cuántos segmentos de huéspedes hay?")
ax.legend()
plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_01_clustering.ipynb)** trae la interpretación de los segmentos, el clustering jerárquico, DBSCAN, el mini-reto de ponerles nombre y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn` y `scipy`, el escalado, los tres algoritmos de agrupación (`KMeans`, `AgglomerativeClustering`, `DBSCAN`), la silueta y el dendrograma.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Fija la semilla `SEMILLA = 42` (para que todos obtengan los mismos grupos) y lee `huespedes.csv` en `huespedes`: 1 200 huéspedes con un identificador y 7 variables de comportamiento.
- **Qué debe salir:**
    - `(1200, 8)` (1 200 filas y 8 columnas) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Escalar: que todas las variables «pesen» lo mismo

#### A2 · Ejemplo — Las primeras filas de huespedes

**Contexto.** No hay etiquetas: nadie nos dice a qué grupo pertenece cada huésped. Es aprendizaje **no supervisado**: dejamos que los datos se agrupen solos y luego **nosotros** interpretamos qué significa cada grupo. Cada fila es un huésped con 7 variables de comportamiento (más su identificador).

- **Qué hace:**
    - `huespedes.head()` muestra las 5 primeras filas de la tabla.
- **Qué debe salir:**
    - Una tabla con 5 filas y 8 columnas: `id_huesped`, `edad`, `gasto_medio_dia`, `noches_medias`, `reservas_anio`, `antelacion_media`, `pct_gasto_actividades` y `viaja_con_ninos`. El primer huésped tiene 26 años, gasta 92,25 € al día y se aloja 3,9 noches.
- **Cómo leerlo:**
    - Cada fila es un huésped y cada columna una variable de su comportamiento. `viaja_con_ninos` vale 0 ó 1; `pct_gasto_actividades`, de 0 a 100.
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas columnas tiene la tabla y cuáles son numéricas?* → **8** columnas; las 8 son numéricas (una es el identificador, que no describe el comportamiento).
2. *¿Qué variable solo puede valer 0 ó 1?* → `viaja_con_ninos`: 1 si el huésped viaja con niños y 0 si no.
3. *¿Hay alguna columna que diga a qué grupo pertenece el huésped?* → **No**: no hay etiquetas. Por eso es aprendizaje **no supervisado**: los grupos los tendrá que encontrar el algoritmo.

</details>

#### A3 · Ejemplo — Las escalas de las variables

Fíjate en las **escalas**: la edad va de 18 a 75, el gasto diario de 35 a 340 y `viaja_con_ninos` solo vale 0 ó 1.

- **Qué hace:**
    - `huespedes.describe()` calcula, para cada columna, el recuento, la media, la desviación típica, el mínimo, los cuartiles y el máximo; `.round(1)` los deja con un decimal y `.T` traspone la tabla (una fila por variable).
- **Qué debe salir:**
    - Una tabla con una fila por variable. La media y la desviación típica de las más relevantes: `edad` 41,2 y 12,1 · `gasto_medio_dia` 124,2 y 60,9 · `antelacion_media` 78,7 y 36,5 · `viaja_con_ninos` 0,3 y 0,5.
- **Cómo leerlo:**
    - Compara la columna `std` (cuánto varía cada variable): el gasto y la antelación varían decenas de unidades; los niños, medio punto.
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué valores va `gasto_medio_dia`?* → De **34,6** a **338,7** euros al día (con una media de 124,2).
2. *¿Qué variable varía más en unidades absolutas y cuál menos?* → `gasto_medio_dia` y `antelacion_media` son las que más varían (`std` 60,9 y 36,5); `viaja_con_ninos` la que menos (0,5).
3. *Si agrupamos por distancia sin tocar estas escalas, ¿qué variables mandarían?* → Las de **números grandes**, es decir el gasto y la antelación; `viaja_con_ninos` apenas contaría. Por eso hay que escalar.

</details>

#### A4 · Ejemplo — Estandarizar una sola columna

**Idea.** Clustering agrupa por **distancia**. Si una variable tiene números grandes (gasto en euros) y otra pequeños (0/1 de niños), la grande manda y la pequeña no cuenta. Solución: **estandarizar** (restar la media y dividir por la desviación típica) para que todas tengan media 0 y desviación 1.

- **Qué hace:**
    - `edad = huespedes["edad"]` toma la columna de edades.
    - `(edad - edad.mean()) / edad.std()` resta la media y divide entre la desviación típica: es la estandarización a mano.
    - El `print` comprueba que la media vale 0 y la desviación 1, redondeadas a 3 decimales.
- **Qué debe salir:**
    - `media: 0.0 · desviación: 1.0`.
- **Cómo leerlo:**
    - Después de estandarizar, 0 significa «en la media» y 1 «una desviación típica por encima»: la edad ya no se mide en años sino en «desviaciones típicas».
- **Si falla:**
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué media y qué desviación tiene la edad estandarizada?* → Media **0** y desviación **1**.
2. *Un huésped de 53 años (la media es 41,2 y la desviación 12,1), ¿cuánto vale su edad escalada?* → (53 − 41,2) / 12,1 ≈ **+1,0**: está una desviación típica por encima de la media.
3. *¿Qué consigue esto para el clustering?* → Que **todas las variables pesen lo mismo** al calcular distancias, sea cual sea su unidad (años, euros, 0/1).

</details>

#### A5 · Ejercicio 1 — La matriz X estandarizada

- **Qué hace:**
    - `variables` es la lista de nombres de columna sin `id_huesped` (el identificador no describe el comportamiento) ni `grupo` (esa columna se crea más adelante; así puedes repetir esta celda).
    - `StandardScaler().fit_transform(huespedes[variables])` estandariza cada columna (media 0, desviación 1) y devuelve una **matriz** de números (`X`), ya sin nombres de columna.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `X debe tener 1200 filas y 7 columnas` → has dejado `id_huesped` (8 columnas) o has guardado en `X` otra cosa que la matriz estandarizada.
    - `Revisa que estés escalando con StandardScaler` → `X` no tiene media 0 y desviación 1: has pasado los datos sin escalar.
    - `NameError: name 'huespedes' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se quita `id_huesped`?* → Porque es solo un número de identificación: **no describe el comportamiento** y, si lo dejáramos, el algoritmo agruparía por un número que no significa nada.
2. *¿Qué tipo de objeto es `X` y qué ha cambiado respecto a `huespedes`?* → Una **matriz** (array de NumPy) de 1200 × 7, **sin nombres de columna** y con todas las variables estandarizadas. Los nombres se guardan aparte en `variables`.
3. *¿Qué media y desviación tiene cada columna de `X`?* → **Media 0 y desviación 1** (así lo comprueba el `assert`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
variables = [c for c in huespedes.columns if c not in ("id_huesped", "grupo")]   # "grupo" aparece más abajo; así puedes repetir esta celda
X = StandardScaler().fit_transform(huespedes[variables])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. k-means: agrupar alrededor de centros

#### A6 · Ejemplo — k-means con k = 3

**Idea.** Le decimos cuántos grupos queremos (`k`). El algoritmo coloca `k` centros, asigna cada huésped al centro más cercano, recalcula los centros y repite hasta que no cambian. Es rápido y fácil de explicar. Su límite: **tú decides `k`** y prefiere grupos «redondos» y de tamaño parecido.

- **Qué hace:**
    - `KMeans(n_clusters=3, n_init=10, random_state=SEMILLA).fit(X)` agrupa los huéspedes en 3 grupos (`n_init=10` repite el proceso 10 veces desde centros distintos y se queda con el mejor).
    - `np.bincount(km3.labels_)` cuenta cuántos huéspedes hay en cada grupo.
    - `km3.inertia_` es la **inercia**: la suma de las distancias al cuadrado de cada huésped a su centro (cuanto menor, grupos más compactos).
- **Qué debe salir:**
    - `Tamaño de cada grupo: [317 480 403]` e `Inercia (suma de distancias al centro): 3487`.
- **Cómo leerlo:**
    - Los tres números del primer `print` suman 1 200 (317 + 480 + 403): cada huésped está en un solo grupo. La inercia solo sirve para comparar modelos con el mismo `k`: siempre baja al aumentar `k`.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos huéspedes hay en cada uno de los tres grupos?* → **317, 480 y 403** (suman 1 200).
2. *¿Qué decide el usuario en k-means y qué decide el algoritmo?* → El usuario decide **cuántos grupos (`k`)**; el algoritmo decide **quién va a cada grupo** moviendo los centros hasta que se estabilizan.
3. *¿Por qué se pone `random_state=SEMILLA`?* → Para que el resultado sea **reproducible**: k-means empieza con centros al azar, y con la misma semilla todos obtienen los mismos grupos.

</details>

#### A7 · Ejercicio 2 — ¿Cuántos grupos? La silueta de cada k

**El coeficiente de silueta.** Para cada huésped compara *lo cerca que está de su grupo* con *lo cerca que está del grupo vecino*. Va de −1 a 1: **cuanto más alto, grupos más compactos y separados**. Por encima de 0,5 se considera una buena estructura; entre 0,25 y 0,5, aceptable.

- **Qué hace:**
    - `for k in range(2, 9)` prueba de 2 a 8 grupos. Para cada `k`: `KMeans(...).fit_predict(X)` agrupa y devuelve la etiqueta de cada huésped, y `silhouette_score(X, etiquetas)` calcula la silueta media.
    - `siluetas[k] = ...` guarda el resultado en el diccionario (clave = `k`, valor = silueta).
    - `max(siluetas, key=siluetas.get)` devuelve la **clave** con mayor valor, es decir el mejor `k` (`k_mejor`).
- **Qué debe salir:**
    - `k con mejor silueta: 4` y `✔ Ejercicio 2 correcto`. Las siluetas valen: 2 → 0,367 · 3 → 0,415 · **4 → 0,431** · 5 → 0,414 · 6 → 0,326 · 7 → 0,317 · 8 → 0,257.
- **Si falla:**
    - `siluetas debe tener las claves 2, 3, ..., 8` → el diccionario está vacío o el bucle no recorre `range(2, 9)`.
    - `Revisa: k_mejor debería ser el k con la silueta más alta` → `k_mejor` no es una clave del diccionario con mayor silueta (¿has cogido `max(siluetas)`, que da la mayor **clave**, 8, en lugar de usar `key=siluetas.get`?).
    - `NameError: name 'X' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué k tiene la mejor silueta y cuánto vale?* → **k = 4**, con una silueta de **0,431**.
2. *¿Cómo de buena es esa estructura según la escala del texto?* → **Aceptable** (entre 0,25 y 0,5): hay grupos, pero no están muy separados; no llega a 0,5.
3. *¿Por qué `max(siluetas)` no daría el mejor k?* → Porque `max` sobre un diccionario compara las **claves** (2 a 8) y devolvería 8. Hay que decirle que compare los **valores**: `key=siluetas.get`.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
siluetas = {}
for k in range(2, 9):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit_predict(X)
    siluetas[k] = silhouette_score(X, etiquetas)
k_mejor = max(siluetas, key=siluetas.get)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Dibujar la silueta de cada k

- **Qué hace:**
    - `ax.plot(list(siluetas), list(siluetas.values()), marker="o")` dibuja la silueta (eje y) frente a `k` (eje x), con un punto por cada `k`.
    - `ax.axvline(k_mejor, ...)` traza una línea roja discontinua en el mejor `k`.
    - El resto pone títulos, etiquetas de ejes y leyenda (`plt.show()` muestra la figura).
- **Qué debe salir:**
    - Una curva con 7 puntos: sube de 0,37 (k = 2) a 0,43 (k = 4), baja a 0,41 (k = 5) y cae hasta 0,26 (k = 8). La línea roja marca k = 4.
- **Cómo leerlo:**
    - Se busca el **máximo** de la curva. Aquí es poco marcado: k = 3, 4 y 5 dan siluetas parecidas (0,415, 0,431 y 0,414); a partir de 6 hay una caída clara.
- **Si falla:**
    - `NameError: name 'siluetas' is not defined` → falta ejecutar el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué k está el máximo de la curva?* → En **k = 4** (silueta 0,431).
2. *¿Con 3, 4 o 5 grupos las siluetas son muy distintas?* → **No**: 0,415, 0,431 y 0,414. La diferencia es pequeña, así que elegir entre 3, 4 o 5 es una decisión que también puede apoyarse en el negocio.
3. *¿Qué pasa a partir de k = 6?* → La silueta **cae** claramente (0,326, 0,317, 0,257): con tantos grupos se parten grupos que en realidad eran uno solo.

</details>